In [6]:
import pandas as pd
import altair as alt
import os
import re
import numpy as np

analysis_date = "261008"

condition_name_mapping = {
    'treatment_recommendation': 'Recommended Treatment',
    'best_options': 'Options to Consider',
    'worst_options': 'Options to Avoid',
}

In [7]:
# CSV file available upon request
df = pd.read_csv("data/raw_respondent_data.csv")

df

,Duration (in seconds),ResponseId,Q3.2,Q3.3,Q5.1_First Click,Q5.1_Last Click,Q5.1_Page Submit,Q5.1_Click Count,Q5.3,Q5.4,...,MargaretNum,MikeShown,MikeNum,specialty,specialty_10_TEXT,practice_setting,clinical_time,Source,specialty_7_TEXT,ethnicity
0,773,eeb889a0dcc3ceec26541806ac31c2b3a4c802d2cebb58...,> 10 years,Proficient - deep understanding,NaN,NaN,NaN,NaN,NaN,NaN,...,2.0,1.0,1.0,"Internal medicine, pulmonary",NaN,"Academic, university affiliated",More than 50% but less than 95%,Mailing List,NaN,NaN
1,831,1937ec6b1aaecc014f3c52464ec060840885cf55130a04...,5-10 years,Proficient - deep understanding,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,1.0,4.0,"Internal medicine, not pulmonary",NaN,"Academic, not university affiliated",All or almost all (95% or more),Mailing List,NaN,NaN
2,154,ed2a0e8cee47108534b2177222f9b8fec48f38508a3803...,> 10 years,Expert - authoritative knowledge,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,1.0,3.0,"Internal medicine, pulmonary",NaN,"Academic, university affiliated",All or almost all (95% or more),Mailing List,NaN,NaN
3,246,4a5f44bf3bc52280a80a570e6b1338c0dfb775995da715...,> 10 years,Expert - authoritative knowledge,NaN,NaN,NaN,NaN,NaN,NaN,...,5.0,1.0,1.0,"Internal medicine, pulmonary",NaN,Community,More than 50% but less than 95%,Mailing List,NaN,NaN
4,940,f610bc27d9c89163b55a6cd3d31af9a8769f5c98f2aa79...,> 10 years,Proficient - deep understanding,NaN,NaN,NaN,NaN,NaN,NaN,...,3.0,1.0,0.0,"Internal medicine, pulmonary",NaN,"Academic, university affiliated",More than 50% but less than 95%,Mailing List,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
223,1325,31eb2234cf64e435644038299c6badddd154308ca2a0b6...,5-10 years,Proficient - deep understanding,NaN,NaN,NaN,NaN,NaN,NaN,...,3.0,NaN,NaN,"Internal medicine, pulmonary",NaN,"Academic, university aﬃliated",Less than 50%,Faculty List,NaN,Yes
224,832,51bdff62f1dba8d9f2f3e0185f1f39c7877fc8fb94e896...,> 10 years,Expert - authoritative knowledge,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,"Internal medicine, pulmonary",NaN,"Academic, university aﬃliated",More than 50% but less than 95%,Faculty List,NaN,No
225,2063,7c180f220940c7ca1c03b1ab615d02ba74404840ce3273...,> 10 years,Proficient - deep understanding,NaN,NaN,NaN,NaN,NaN,NaN,...,2.0,1.0,0.0,"Internal medicine, pulmonary",NaN,"Academic, university aﬃliated",Less than 50%,Faculty List,NaN,No
226,1061,f579ef612f09b1585e7aa31472b999046c81a0898838bf...,5-10 years,Proficient - deep understanding,NaN,NaN,NaN,NaN,NaN,NaN,...,3.0,1.0,1.0,"Internal medicine, pulmonary",NaN,"Academic, university aﬃliated",Less than 50%,Faculty List,NaN,No


In [8]:
df['Q17.1'].value_counts()

Q17.1
Yes, I'll review two additional cases    143
No, take me to the final questions        84
Name: count, dtype: int64

## Process Respondent-Level Data

In [9]:
# Clean demographic columns
df['practice_setting'] = df['practice_setting'].replace({
    'Academic, university aﬃliated': 'Academic, university affiliated',
    'Academic, not university aﬃliated': 'Academic, not university affiliated'
})
df['specialty'] = df['specialty'].where(~df['specialty'].str.startswith('Other'), 'Other')
df['clinical_time'] = df['clinical_time'].replace({
    'More than 50% but less than 95%': 'Majority',
    'Less than 50%': 'Minority',
    'All or almost all (95% or more)': 'Full',
})
df['Duration (in seconds)'] = df['Duration (in seconds)'].astype(float)

In [10]:
perception_cols = ['Productivity',
    'Effectiveness',
    'EaseOfUse',
    'HighOutputQuality',
    'WouldUse']
df = df.rename(columns={
    'Q3.2': 'Experience',
    'Q3.3': 'SepsisExpertise',
    'Q18.2_1': 'Productivity',
    'Q18.2_2': 'Effectiveness',
    'Q18.2_3': 'EaseOfUse',
    'Q18.2_4': 'HighOutputQuality',
    'Q18.2_5': 'WouldUse',
    'Q18.4': 'UsefulnessFreeText',
    'Q18.5': 'IdealAIRoleFreeText',
    'Q18.6': 'FeedbackFreeText',
})
df[perception_cols] = df[perception_cols].replace({
    'Strongly disagree': 1, 'Disagree': 2, 'Somewhat disagree': 3, 'Neither agree nor disagree': 4,
    'Somewhat agree': 5, 'Agree': 6, 'Strongly agree': 7
})
    
df['Total Satisfaction'] = df[perception_cols].sum(axis=1)

In [11]:
print(df['specialty'].value_counts())
print(df['specialty_10_TEXT'].value_counts())
print(df['clinical_time'].value_counts())
print(df['practice_setting'].value_counts())
print(df['Experience'].value_counts())
print(df['SepsisExpertise'].value_counts())

specialty
Internal medicine, pulmonary        160
Internal medicine, not pulmonary     16
Anesthesiology                       15
Emergency medicine                   14
Other                                14
Surgery                               8
Neurology                             1
Name: count, dtype: int64
specialty_10_TEXT
Critical Care Medicine                         1
pulmonary and critical care fellow             1
Emergency and internal medicine                1
Internal Medicine-Pediatrics                   1
Pulmonary/Critical Care                        1
Pulmonary Critical Care                        1
Critical Care                                  1
Ob/gyn                                         1
critical care medicine/ infectious diseases    1
Internal Medicine, Critical Care               1
Critical care and nephrology                   1
Name: count, dtype: int64
clinical_time
Minority    87
Majority    76
Full        62
Name: count, dtype: int64
practice_setting

In [12]:
# Clean multiple choice demographics
df['Experience'] = df['Experience'].replace({
    '< 1 year': 1,
    '1-2 years': 2,
    '3-5 years': 3,
    '5-10 years': 4,
    '> 10 years': 5
})
df['SepsisExpertise'] = df['SepsisExpertise'].replace({
    'Novice - no experience': 1,
    'Beginner - some working knowledge': 2,
    'Competent - good working and background knowledge': 3,
    'Proficient - deep understanding': 4,
    'Expert - authoritative knowledge': 5
})
print(df['Experience'].value_counts())
print(df['SepsisExpertise'].value_counts())

Experience
5    178
4     39
3     10
1      1
Name: count, dtype: int64
SepsisExpertise
4    105
5    105
3     18
Name: count, dtype: int64


In [13]:
df.columns.tolist()

['Duration (in seconds)',
 'ResponseId',
 'Experience',
 'SepsisExpertise',
 'Q5.1_First Click',
 'Q5.1_Last Click',
 'Q5.1_Page Submit',
 'Q5.1_Click Count',
 'Q5.3',
 'Q5.4',
 'Q5.5',
 'Q6.1_First Click',
 'Q6.1_Last Click',
 'Q6.1_Page Submit',
 'Q6.1_Click Count',
 'Q6.3',
 'Q6.4',
 'Q6.5',
 'Q6.7',
 'Q6.8',
 'Q6.8_8_TEXT',
 'Q7.1_First Click',
 'Q7.1_Last Click',
 'Q7.1_Page Submit',
 'Q7.1_Click Count',
 'Q7.3',
 'Q7.4',
 'Q7.5',
 'Q8.1_First Click',
 'Q8.1_Last Click',
 'Q8.1_Page Submit',
 'Q8.1_Click Count',
 'Q8.3',
 'Q8.4',
 'Q8.5',
 'Q8.7',
 'Q8.8',
 'Q8.8_8_TEXT',
 'Q9.1_First Click',
 'Q9.1_Last Click',
 'Q9.1_Page Submit',
 'Q9.1_Click Count',
 'Q9.3',
 'Q9.4',
 'Q9.5',
 'Q10.1_First Click',
 'Q10.1_Last Click',
 'Q10.1_Page Submit',
 'Q10.1_Click Count',
 'Q10.3',
 'Q10.4',
 'Q10.5',
 'Q10.7',
 'Q10.8',
 'Q10.8_8_TEXT',
 'Q11.1_First Click',
 'Q11.1_Last Click',
 'Q11.1_Page Submit',
 'Q11.1_Click Count',
 'Q11.3',
 'Q11.4',
 'Q11.5',
 'Q12.1_First Click',
 'Q12.1_Last 

In [14]:
no_influences = df[[c for c in df.columns if c.endswith(".8")]].apply(lambda x: x.str.contains("did not affect my thinking at all", case=False), axis=0)
df['SelfReportedUnaffected'] = no_influences.sum(axis=1) >= (~pd.isna(no_influences)).sum(axis=1) - 1
df['SelfReportedAffected'] = no_influences.sum(axis=1) == 0
df['SelfReportedUnaffected'].mean(), df['SelfReportedAffected'].mean()

(np.float64(0.07894736842105263), np.float64(0.4780701754385965))

In [19]:
df.assign(Condition=df['AIInterface'])[[
    'ResponseId',
    'AIInterface',
    'specialty',
    'specialty_10_TEXT',
    'practice_setting',
    'clinical_time',
    'specialty_7_TEXT',
    'ethnicity',
    'Duration (in seconds)',
    'Experience',
    'SepsisExpertise',
    'Productivity',
    'Effectiveness',
    'EaseOfUse',
    'HighOutputQuality',
    'WouldUse',
    'Total Satisfaction',
    'SelfReportedUnaffected',
    'SelfReportedAffected',
    'UsefulnessFreeText',
    'IdealAIRoleFreeText',
    'FeedbackFreeText']].to_csv(f"data/cleaned_respondent_level_{analysis_date}.csv", index=False)

## Decision-Level Data

In [20]:
df.columns = [re.sub(r"^Q(5|6|7|8|9|10|11|12|13|14|15|16)\.(.*)$", r"Decision_\2_\1", c) for c in df.columns]
long_df = pd.wide_to_long(
    df,
    ['Decision_1_First Click_', 'Decision_1_Last Click_', 'Decision_1_Page Submit_', 'Decision_1_Click Count_',
         'Decision_3_', 'Decision_4_', 'Decision_5_', 'Decision_7_', 'Decision_8_', 'Decision_8_8_TEXT_'],
    'ResponseId',
    'Case' 
).reset_index()
long_df.head(10)

,ResponseId,Case,AIInterface,DeborahNum,DeborahShown,Duration (in seconds),EaseOfUse,Effectiveness,EstherNum,EstherShown,...,Decision_1_First Click_,Decision_1_Last Click_,Decision_1_Page Submit_,Decision_1_Click Count_,Decision_3_,Decision_4_,Decision_5_,Decision_7_,Decision_8_,Decision_8_8_TEXT_
0,eeb889a0dcc3ceec26541806ac31c2b3a4c802d2cebb58...,5,Recommended Treatment,0.0,1.0,773.0,5,4,4.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1937ec6b1aaecc014f3c52464ec060840885cf55130a04...,5,Options to Consider,5.0,1.0,831.0,3,2,2.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,ed2a0e8cee47108534b2177222f9b8fec48f38508a3803...,5,Options to Avoid,0.0,1.0,154.0,5,5,1.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,4a5f44bf3bc52280a80a570e6b1338c0dfb775995da715...,5,Options to Consider,2.0,1.0,246.0,6,6,0.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,f610bc27d9c89163b55a6cd3d31af9a8769f5c98f2aa79...,5,Options to Avoid,4.0,1.0,940.0,4,3,5.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,d1c39bac8ff0d66dfe8e791bd97fd3b675cf22fed17098...,5,Recommended Treatment,3.0,1.0,553.0,7,6,1.0,1.0,...,87.773,98.572,98.575,6.0,Give up to 1 L of IV fluid.,Wean the existing vasopressor.,Somewhat confident,NaN,NaN,NaN
6,2c989695ed62cf998ba11914dbb69e9fc0252917a0fc63...,5,Options to Consider,1.0,1.0,1189.0,4,3,2.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,508be831d8076d9c357f0c2418e5719a33f40ba123ef61...,5,Recommended Treatment,0.0,1.0,857.0,2,2,5.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,4e7fe7ad932d6231f16864a7d8f85edb905c5e8e2a1c96...,5,Options to Avoid,5.0,1.0,936.0,4,2,0.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,79894f268118d96d7a197035fbf7313efd1d9fec8e3fac...,5,Options to Avoid,2.0,1.0,816.0,6,1,NaN,NaN,...,95.716,104.711,104.716,4.0,Give neither IV fluids nor diuretics.,Continue the existing vasopressor.,Somewhat confident,NaN,NaN,NaN


In [21]:
influences = long_df['Decision_8_'].str.split(",").explode().unique()
for c in influences:
    if pd.isna(c): continue
    long_df[f'Influence: {c}'] = long_df['Decision_8_'].str.contains(c).fillna(False)

long_df['AIInterface'] = long_df['AIInterface'].where(long_df['Case'] % 2 == 0, 'No AI')
long_df['Case'] = ((long_df['Case'] - 5) // 2).replace({
    0: 'Deborah',
    1: 'Heather',
    2: 'Esther',
    3: 'Richard',
    4: 'Margaret',
    5: 'Mike'
})
long_df

/var/folders/dp/qnkl7j7n113cxdv22y7tr3dr0000gq/T/ipykernel_4564/2071281912.py:4: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  long_df[f'Influence: {c}'] = long_df['Decision_8_'].str.contains(c).fillna(False)


,ResponseId,Case,AIInterface,DeborahNum,DeborahShown,Duration (in seconds),EaseOfUse,Effectiveness,EstherNum,EstherShown,...,Influence: Prompted me to assess the patient further,Influence: Prompted me to consider an alternative plan,Influence: I'm not sure,Influence: Helped me decide between alternatives,Influence: Made me more confident in my plan,Influence: Affected my thinking in some other way (please describe):,Influence: Made me second-guess my plan,Influence: Did not affect my thinking at all,Influence: Gave me an initial plan,Influence: Helped me form an initial assessment of the patient
0,eeb889a0dcc3ceec26541806ac31c2b3a4c802d2cebb58...,Deborah,No AI,0.0,1.0,773.0,5,4,4.0,1.0,...,False,False,False,False,False,False,False,False,False,False
1,1937ec6b1aaecc014f3c52464ec060840885cf55130a04...,Deborah,No AI,5.0,1.0,831.0,3,2,2.0,1.0,...,False,False,False,False,False,False,False,False,False,False
2,ed2a0e8cee47108534b2177222f9b8fec48f38508a3803...,Deborah,No AI,0.0,1.0,154.0,5,5,1.0,1.0,...,False,False,False,False,False,False,False,False,False,False
3,4a5f44bf3bc52280a80a570e6b1338c0dfb775995da715...,Deborah,No AI,2.0,1.0,246.0,6,6,0.0,1.0,...,False,False,False,False,False,False,False,False,False,False
4,f610bc27d9c89163b55a6cd3d31af9a8769f5c98f2aa79...,Deborah,No AI,4.0,1.0,940.0,4,3,5.0,1.0,...,False,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2731,31eb2234cf64e435644038299c6badddd154308ca2a0b6...,Mike,Options to Avoid,2.0,1.0,1325.0,7,2,1.0,1.0,...,False,False,False,False,False,False,False,False,False,False
2732,51bdff62f1dba8d9f2f3e0185f1f39c7877fc8fb94e896...,Mike,Recommended Treatment,1.0,1.0,832.0,4,3,2.0,1.0,...,False,False,False,False,False,False,False,False,False,False
2733,7c180f220940c7ca1c03b1ab615d02ba74404840ce3273...,Mike,Options to Consider,3.0,1.0,2063.0,6,4,5.0,1.0,...,False,False,False,False,False,False,True,False,False,False
2734,f579ef612f09b1585e7aa31472b999046c81a0898838bf...,Mike,Options to Consider,NaN,NaN,1061.0,6,6,2.0,1.0,...,False,False,False,True,False,False,False,False,False,False


In [22]:
nums = []
choice_change_fluids = []
choice_change_vaso = []
for i, row in long_df.iterrows():
    nums.append(row[row['Case'] + 'Num'])
    choice_change_fluids.append(row[f"__js_ChoiceChange{row['Case']}Fluids"])
    choice_change_vaso.append(row[f"__js_ChoiceChange{row['Case']}Vaso"])
    
long_df = long_df.assign(CaseNum=nums, ChoiceChangeFluids=choice_change_fluids, ChoiceChangeVaso=choice_change_vaso)
long_df

,ResponseId,Case,AIInterface,DeborahNum,DeborahShown,Duration (in seconds),EaseOfUse,Effectiveness,EstherNum,EstherShown,...,Influence: Helped me decide between alternatives,Influence: Made me more confident in my plan,Influence: Affected my thinking in some other way (please describe):,Influence: Made me second-guess my plan,Influence: Did not affect my thinking at all,Influence: Gave me an initial plan,Influence: Helped me form an initial assessment of the patient,CaseNum,ChoiceChangeFluids,ChoiceChangeVaso
0,eeb889a0dcc3ceec26541806ac31c2b3a4c802d2cebb58...,Deborah,No AI,0.0,1.0,773.0,5,4,4.0,1.0,...,False,False,False,False,False,False,False,0.0,1.0,1.0
1,1937ec6b1aaecc014f3c52464ec060840885cf55130a04...,Deborah,No AI,5.0,1.0,831.0,3,2,2.0,1.0,...,False,False,False,False,False,False,False,5.0,1.0,2.0
2,ed2a0e8cee47108534b2177222f9b8fec48f38508a3803...,Deborah,No AI,0.0,1.0,154.0,5,5,1.0,1.0,...,False,False,False,False,False,False,False,0.0,1.0,1.0
3,4a5f44bf3bc52280a80a570e6b1338c0dfb775995da715...,Deborah,No AI,2.0,1.0,246.0,6,6,0.0,1.0,...,False,False,False,False,False,False,False,2.0,1.0,NaN
4,f610bc27d9c89163b55a6cd3d31af9a8769f5c98f2aa79...,Deborah,No AI,4.0,1.0,940.0,4,3,5.0,1.0,...,False,False,False,False,False,False,False,4.0,1.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2731,31eb2234cf64e435644038299c6badddd154308ca2a0b6...,Mike,Options to Avoid,2.0,1.0,1325.0,7,2,1.0,1.0,...,False,False,False,False,False,False,False,NaN,NaN,NaN
2732,51bdff62f1dba8d9f2f3e0185f1f39c7877fc8fb94e896...,Mike,Recommended Treatment,1.0,1.0,832.0,4,3,2.0,1.0,...,False,False,False,False,False,False,False,NaN,NaN,NaN
2733,7c180f220940c7ca1c03b1ab615d02ba74404840ce3273...,Mike,Options to Consider,3.0,1.0,2063.0,6,4,5.0,1.0,...,False,False,False,True,False,False,False,0.0,1.0,1.0
2734,f579ef612f09b1585e7aa31472b999046c81a0898838bf...,Mike,Options to Consider,NaN,NaN,1061.0,6,6,2.0,1.0,...,True,False,False,False,False,False,False,1.0,2.0,1.0


In [23]:
long_df = long_df[[
    'ResponseId', 'Case', 'AIInterface', 'CaseNum', 'Decision_1_First Click_',
       'Decision_1_Last Click_', 'Decision_1_Page Submit_',
       'Decision_1_Click Count_', 'Decision_3_', 'Decision_4_', 'Decision_5_',
       'Decision_7_', 'Decision_8_8_TEXT_', 'ChoiceChangeFluids', 'ChoiceChangeVaso'
] + [c for c in long_df.columns if c.startswith("Influence:")]]
long_df = long_df[~pd.isna(long_df['Decision_3_'])]

In [24]:
alt.Chart(long_df).mark_bar().encode(
    alt.X("Decision_1_Page Submit_:Q", bin=alt.Bin(extent=[0, 600], steps=[5, 10, 15])),
    alt.Y("count():Q")
)

alt.Chart(...)

In [25]:
long_df = long_df.rename(columns={
    "Q3.2": "Experience",
    "Q3.3": "SepsisExpertise",
    "Decision_1_First Click_": "First Click Time",
    "Decision_1_Last Click_": "Last Click Time",
    "Decision_1_Page Submit_": "Page Submit Time",
    "Decision_1_Click Count_": "Click Count",
    "Decision_3_": "Fluid",
    "Decision_4_": "Vasopressor",
    "Decision_5_": "Confidence",
    "Decision_7_": "AIQuality",
    "Decision_8_8_TEXT_": "Other Influences",
})

## Add Decision Quality

In [26]:
fluid_terms = {
    'Give neither IV fluids nor diuretics.': 0,
    "Stop diuretics and don't give IV fluids.": 0,
    'Give up to 1 L of IV fluid.': 1,
    'Stop diuretics and give up to 1 L of IV fluid.': 1,
    'Give more than 1 L of IV fluid.': 2,
    'Stop diuretics and give more than 1 L of IV fluid.': 2,
    'Give a diuretic.': 3,
    'Continue the diuretic.': 3
}

vaso_terms = {
    'Do not give vasopressors.': 0,
    'Wean both existing vasopressors.': 0,
    'Wean the existing vasopressor.': 0,
    'Continue the existing vasopressor.': 1,
    'Start a vasopressor.': 1,
    'Wean the secondary vasopressor.': 1, 
    'Add another vasopressor.': 2,
    'Continue both vasopressors.': 2,
    'Start multiple vasopressors.': 2,
    'Wean the existing vasopressor ONLY IF I ordered fluid and there is a positive response in less than four hours.': 4,
    'Wean one or both vasopressors ONLY IF I ordered fluid and there is a positive response in less than four hours.': 4,
    'Start a vasopressor ONLY IF I ordered fluid and there is NO positive response in less than four hours.': 3,
    'Add another vasopressor ONLY IF I ordered fluid and there is NO positive response in less than four hours.': 5,
    'Wean the secondary vasopressor now, and wean the other vasopressor ONLY IF I ordered fluid and there is a positive response in less than four hours.': 5,
}
print(set(long_df['Fluid'].unique()) - set(fluid_terms.keys()))
print(set(long_df['Vasopressor'].unique()) - set(vaso_terms.keys()))

fluid_nums = long_df['Fluid'].replace(fluid_terms)
vaso_nums = long_df['Vasopressor'].replace(vaso_terms)

invalid_decision = (vaso_nums >= 3) & ~fluid_nums.isin([1, 2])
print("invalid decision:", invalid_decision.sum())

long_df = long_df[~invalid_decision]
fluid_nums = fluid_nums[~invalid_decision]
vaso_nums = vaso_nums[~invalid_decision]

base_decision_numbers = fluid_nums * 3 + vaso_nums
base_decision_numbers = base_decision_numbers.where(vaso_nums < 3,
                                                    12 + (fluid_nums - 1) + (vaso_nums - 3))
base_decision_numbers = base_decision_numbers.where(vaso_nums < 4,
                                                    12 + (fluid_nums - 1) * 2 + (vaso_nums - 4))
base_decision_numbers += 1
base_decision_numbers.value_counts()

set()
set()
invalid decision: 19


2     398
13    178
11    143
5     135
14     74
3      66
1      48
8      33
12     28
10     21
4      15
15     11
16     10
6       8
9       6
7       5
Name: count, dtype: int64

In [27]:
decision_num_df = pd.concat([long_df, fluid_nums.rename('fluid_num'), vaso_nums.rename('vaso_num'), base_decision_numbers.rename('decision_num')], axis=1)
for num in sorted(decision_num_df['decision_num'].unique()):
    print(num)
    print(decision_num_df[decision_num_df['decision_num'] == num][['Fluid', 'Vasopressor', 'fluid_num', 'vaso_num']].value_counts())

1
Fluid                                     Vasopressor                       fluid_num  vaso_num
Give neither IV fluids nor diuretics.     Do not give vasopressors.         0          0           21
                                          Wean the existing vasopressor.    0          0           16
Stop diuretics and don't give IV fluids.  Do not give vasopressors.         0          0            7
Give neither IV fluids nor diuretics.     Wean both existing vasopressors.  0          0            4
Name: count, dtype: int64
2
Fluid                                     Vasopressor                         fluid_num  vaso_num
Give neither IV fluids nor diuretics.     Start a vasopressor.                0          1           243
Stop diuretics and don't give IV fluids.  Start a vasopressor.                0          1            70
Give neither IV fluids nor diuretics.     Continue the existing vasopressor.  0          1            48
                                          Wean the se

In [30]:
quality_df = pd.read_csv("data/all_quality_ratings.csv").rename(columns={
    'Final': 'QualityScore',
    'E1': 'QualityScore_E1',
    'E2': 'QualityScore_E2',
    'E3': 'QualityScore_E3',
    'E4': 'QualityScore_E4',
    'E5': 'QualityScore_E5',
})

In [31]:
scored_df = pd.merge(decision_num_df, quality_df, how='left', left_on=['Case', 'decision_num'], right_on=['Case', 'Decision']).drop(columns=['Decision'])
scored_df

,ResponseId,Case,AIInterface,CaseNum,First Click Time,Last Click Time,Page Submit Time,Click Count,Fluid,Vasopressor,...,Influence: Helped me form an initial assessment of the patient,fluid_num,vaso_num,decision_num,QualityScore_E1,QualityScore_E2,QualityScore_E3,QualityScore_E4,QualityScore_E5,QualityScore
0,d1c39bac8ff0d66dfe8e791bd97fd3b675cf22fed17098...,Deborah,No AI,3.0,87.773,98.572,98.575,6.0,Give up to 1 L of IV fluid.,Wean the existing vasopressor.,...,False,1,0,4,2,4,4,5,3,4
1,79894f268118d96d7a197035fbf7313efd1d9fec8e3fac...,Deborah,No AI,2.0,95.716,104.711,104.716,4.0,Give neither IV fluids nor diuretics.,Continue the existing vasopressor.,...,False,0,1,2,4,7,3,2,7,4
2,731eae813be1216914484db532c9c03faf42712e73dd86...,Deborah,No AI,3.0,79.675,87.921,87.928,4.0,Give up to 1 L of IV fluid.,Add another vasopressor ONLY IF I ordered flui...,...,False,1,5,14,6,6,4,3,4,4
3,839dadbd27a88008513527da23d576efcfcaee434141bf...,Deborah,No AI,0.0,23.235,43.681,43.694,4.0,Give up to 1 L of IV fluid.,Wean the existing vasopressor.,...,False,1,0,4,2,4,4,5,3,4
4,a79fc41eabbcd9b23c3a0a7cd9927b8c291f556899c41d...,Deborah,No AI,1.0,22.559,108.769,108.784,15.0,Give up to 1 L of IV fluid.,Add another vasopressor ONLY IF I ordered flui...,...,False,1,5,14,6,6,4,3,4,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1174,d37f8c51e5c4c5fd5be4e834f2ea572fa4d9e87827feb5...,Mike,Options to Consider,1.0,168.603,180.971,180.979,4.0,Give neither IV fluids nor diuretics.,Start a vasopressor.,...,True,0,1,2,5,6,6,6,6,6
1175,0921acf396f9251a698f8cb8a3353d90da2f252b4046d8...,Mike,Recommended Treatment,2.0,4.473,603.355,603.363,14.0,Give up to 1 L of IV fluid.,Start a vasopressor ONLY IF I ordered fluid an...,...,False,1,3,13,7,5,5,5,4,5
1176,7c180f220940c7ca1c03b1ab615d02ba74404840ce3273...,Mike,Options to Consider,0.0,39.706,131.113,131.131,5.0,Give up to 1 L of IV fluid.,Start a vasopressor.,...,False,1,1,5,7,6,5,6,5,6
1177,f579ef612f09b1585e7aa31472b999046c81a0898838bf...,Mike,Options to Consider,1.0,27.718,182.456,182.476,8.0,Give a diuretic.,Start a vasopressor.,...,False,3,1,11,4,3,5,5,3,4


In [32]:
scored_df['First Click Time'] = scored_df['First Click Time'].astype(float)
scored_df['Last Click Time'] = scored_df['Last Click Time'].astype(float)
scored_df['Page Submit Time'] = scored_df['Page Submit Time'].astype(float)

scored_df['TimeSpent'] = scored_df['Page Submit Time']
scored_df['LongTime'] = scored_df['Page Submit Time'] >= 120 # scored_df['Page Submit Time'].median()

scored_df['InfluenceGroupAssessment'] = scored_df[['Influence: Prompted me to assess the patient further',
                                              'Influence: Helped me form an initial assessment of the patient']].any(axis=1)
scored_df['InfluenceGroupPlansConsidered'] = scored_df[['Influence: Gave me an initial plan',
                                                   'Influence: Prompted me to consider an alternative plan']].any(axis=1)
scored_df['InfluenceGroupPlansEvaluated'] = scored_df[['Influence: Helped me decide between alternatives']].any(axis=1)
scored_df['InfluenceGroupConfidence'] = scored_df[['Influence: Made me more confident in my plan',
                                                    'Influence: Made me second-guess my plan']].any(axis=1)
influence_columns = ['Influence: Prompted me to assess the patient further',
 'Influence: Prompted me to consider an alternative plan',
 "Influence: I'm not sure",
 'Influence: Made me more confident in my plan',
 'Influence: Affected my thinking in some other way (please describe):',
 'Influence: Made me second-guess my plan',
 'Influence: Did not affect my thinking at all',
 'Influence: Gave me an initial plan',
 'Influence: Helped me decide between alternatives',
 'Influence: Helped me form an initial assessment of the patient']
short_names = [
    'InfluenceAssessFurther',
    'InfluenceConsiderAlternative',
    "InfluenceImNotSure",
    'InfluenceMoreConfident',
    'InfluenceOther',
    'InfluenceSecondGuess',
    'InfluenceNoEffect',
    'InfluenceInitialPlan',
    'InfluenceHelpDecide',
    'InfluenceInitialAssessment'
]
scored_df = scored_df.rename(columns=dict(zip(influence_columns, short_names)))

In [33]:
scored_df.to_csv(f"data/cleaned_decision_level_{analysis_date}.csv", index=False)

# Free-Text Responses

In [ ]:
import re
df = pd.read_csv("data/cleaned_respondent_level_251204.csv")

print(len(df['UsefulnessFreeText'][df['UsefulnessFreeText'].fillna('') != '']), "out of", len(df))
useful_responses = df['UsefulnessFreeText'][df['UsefulnessFreeText'].fillna('') != '']
lengths = useful_responses.apply(lambda r: len(re.split(r'\W+', r)))
print(lengths.median(), lengths.min(), lengths.max())
for ai in df['AIInterface'].unique():
    print(ai)
    print("USEFULNESS\n\n")
    print('\n\n'.join(df[df['AIInterface'] == ai]['UsefulnessFreeText'][df['UsefulnessFreeText'].fillna('') != ''].values.tolist()))
    print("\n\n")
    print("IDEAL ROLE\n\n")
    print('\n\n'.join(df[df['AIInterface'] == ai]['IdealAIRoleFreeText'][df['IdealAIRoleFreeText'].fillna('') != ''].values.tolist()))
    print("\n\n")
    print("MISC\n\n")
    print('\n\n'.join(df[df['AIInterface'] == ai]['FeedbackFreeText'][df['FeedbackFreeText'].fillna('') != ''].values.tolist()))
    print("\n\n")